#### 제 9회 기출


##### 작업형 제 1유형


In [213]:
# 1
import pandas as pd

df = pd.read_csv('loan.csv')

df['총대출액'] = df['신용대출'] + df['담보대출']
new_df = df.pivot_table(index='지역코드', columns='성별', values='총대출액', aggfunc='sum')
new_df['sum'] = abs(new_df[1] - new_df[2])
new_df['sum'].idxmax()


np.int64(4100000278)

In [214]:
# 2-1
import pandas as pd

df = pd.read_csv('crime.csv')
df1 = df.iloc[:,2:][df['구분'] == '발생건수'].reset_index(drop=True)
df2 = df.iloc[:,2:][df['구분'] == '검거건수'].reset_index(drop=True)
df3 = df2 / df1

listname = df3.idxmax(axis=1)

result = 0
for i, item in enumerate(listname) :
    result = result + df2.loc[i, item]

result


np.int64(7799)

In [215]:
# 2-2
df = pd.melt(df, id_vars=['연도','구분'], var_name='범죄유형', value_name='건수')

new_df = df.pivot_table(index=['연도','범죄유형'], columns='구분', values='건수', aggfunc='sum').reset_index()
new_df['검거율'] = new_df['검거건수'] / new_df['발생건수']
listname = new_df.groupby('연도')['검거율'].idxmax()
new_df.loc[listname]['검거건수'].sum()

np.int64(7799)

In [216]:
# 3
import pandas as pd

df = pd.read_csv('hr.csv')
df['만족도'] = df['만족도'].fillna(df['만족도'].mean())
df_mean = df.groupby(['부서','성과등급'])['근속연수'].transform('mean').astype(int)
df['근속연수'] = df['근속연수'].fillna(df_mean)

df['value'] = df['연봉'] / df['근속연수']
df.sort_values('value', ascending=False).reset_index(drop=True) # 1
df['value2'] = df['연봉'] / df['만족도']
df.sort_values('value2', ascending=False).reset_index(drop=True) # 6

,사원번호,부서,성과등급,연봉,근속연수,교육참가횟수,만족도,value,value2
0,E0491,Finance,B,149000000,13.0,6,1.0,1.146154e+07,1.490000e+08
1,E0738,HR,A,146200000,9.0,6,1.0,1.624444e+07,1.462000e+08
2,E0409,Finance,B,146100000,6.0,3,1.0,2.435000e+07,1.461000e+08
3,E0576,Sales,B,144500000,9.0,1,1.0,1.605556e+07,1.445000e+08
4,E0976,Manager,C,142500000,11.0,3,1.0,1.295455e+07,1.425000e+08
...,...,...,...,...,...,...,...,...,...
995,E0236,Finance,A,46800000,12.0,1,9.0,3.900000e+06,5.200000e+06
996,E0848,HR,A,41000000,19.0,6,8.0,2.157895e+06,5.125000e+06
997,E0828,Manager,A,45300000,18.0,1,9.0,2.516667e+06,5.033333e+06
998,E0930,Sales,C,40000000,11.0,6,8.0,3.636364e+06,5.000000e+06


---

##### 작업형 제 2유형

In [2]:
import pandas as pd 

train = pd.read_csv('farm_train.csv')
test = pd.read_csv('farm_test.csv')

# train.info()
# test.info()
# print(train.shape, test.shape) # (4000, 9) (1000, 8)
# print(train.isnull().sum())
# print(test.isnull().sum())
# train.describe()
# train.describe(include='O')

cols = train.select_dtypes(include='O').columns
for col in cols :
    train_set = set(train[col])
    test_set = set(test[col])
    if train_set == test_set :
        print(col, '동일')
    else :
        print(col, '동일하지 않음')

print('======================================')

y_train = train.pop('농약검출여부')

# 원-핫 인코딩
# train = pd.get_dummies(train)
# test = pd.get_dummies(test)
# train.shape, train_oh.shape, test.shape, test_oh.shape

# 라벨 인코딩
from sklearn.preprocessing import LabelEncoder
for col in cols :
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    
# 스케일링
from sklearn.preprocessing import RobustScaler
cols_num = train.select_dtypes(exclude='O').columns
mm = RobustScaler()
train[cols_num] = mm.fit_transform(train[cols_num])
test[cols_num] = mm.transform(test[cols_num])

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, y_train, test_size=0.2, random_state=0)

# from sklearn.ensemble import RandomForestClassifier
# rf = RandomForestClassifier(random_state=0, max_depth=21)
# rf.fit(X_train, y_train)
# y_pred = rf.predict(X_val)
# print(rf.classes_)
# print(y_pred)

import lightgbm as lgb
lgbm = lgb.LGBMClassifier(random_state=0, verbose=-1)
lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_val)

from sklearn.metrics import f1_score
f1 = f1_score(y_val, y_pred, average='macro')
print('f1-score:', f1) # 0.8532014300116062 # 0.8868077979576238

pred = lgbm.predict(test)
result = pd.DataFrame({'pred':pred}).to_csv('result.csv', index=False)

pd.read_csv('result.csv').head()



지역 동일
작물종류 동일
토양유형 동일
등급 동일
f1-score: 0.9256154893769262


,pred
0,2
1,0
2,0
3,2
4,0


---

##### 작업형 제 3유형

In [ ]:
# 1
import pandas as pd
from statsmodels.formula.api import ols
from sklearn.metrics import root_mean_squared_error

df = pd.read_csv('design.csv')

train = df.iloc[:140].copy()
test = df.iloc[140:].copy()

model = ols('design ~ c1 + c2 + c3 + c4 + c5', data=train).fit()
print(model.summary()) # p-value < 0.05 인 독립변수 3개

print('=======================================')

new_model = ols('design ~ c1 + c2 + c4', data=train).fit()
train['pred'] = new_model.predict(train)

print('상관계수:', round(train['pred'].corr(train['design']), 3))

print('=======================================')

test['pred'] = new_model.predict(test)

rmse = root_mean_squared_error(test['pred'], test['design'])
print('RMSE:', round(rmse, 3))




                            OLS Regression Results                            
Dep. Variable:                 design   R-squared:                       0.266
Model:                            OLS   Adj. R-squared:                  0.238
Method:                 Least Squares   F-statistic:                     9.697
Date:                Thu, 19 Jun 2025   Prob (F-statistic):           6.37e-08
Time:                        14:43:36   Log-Likelihood:                -468.72
No. Observations:                 140   AIC:                             949.4
Df Residuals:                     134   BIC:                             967.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     53.0220      2.294     23.112      0.0

In [ ]:
# 2
import pandas as pd
from statsmodels.formula.api import logit
import numpy as np

df = pd.read_csv('retention.csv')

model = logit('Churn ~ MonthlyCharges + CustomerTenure + HasPhoneService + HasTechInsurance', data=df).fit()
print(model.summary())
print('=======================================')
print('p-value:', round(model.pvalues['MonthlyCharges'], 3))
print('=======================================')
print('odds ratio:', round(np.exp(model.params['HasPhoneService']), 3))
print('=======================================')
pred = model.predict(df)
sum(pred > 0.3)

Optimization terminated successfully.
         Current function value: 0.582234
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                  Churn   No. Observations:                   80
Model:                          Logit   Df Residuals:                       75
Method:                           MLE   Df Model:                            4
Date:                Thu, 19 Jun 2025   Pseudo R-squ.:                  0.1585
Time:                        15:51:52   Log-Likelihood:                -46.579
converged:                       True   LL-Null:                       -55.352
Covariance Type:            nonrobust   LLR p-value:                  0.001513
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -4.4731      1.437     -3.114      0.002      -7.289      -1.657
MonthlyChar

65